# P3 - Information Leakage across Adaptation Techniques
In this notebook, we explore information leakage across tasks across a range of models and adaptation techniques. The models we will be working with are `Llama3:8b`, `Gemma3:4b` and `DeepSeek-R1:8b`. The models will each be exposed to factually incorrect data through a Machine Translation (MT) task (English to Swahili), using 3 separate adaptation techniques: In-context Learning (ICL), Supervised Fine-tuning (SFT) and Low-Rank Adaptation (LoRA). The goal is to determine if information used for adaptation in one task (MT) may leak into other unrelated tasks, in this case Question Answering (QA).

To achieve this, we use the Swahili subset from the SmolDoc dataset (Caswell et. al, 2025), which contains generated documents, each with an associated factuality annotation. By combining the two, we can determine if information has leaked, since we can look for the factual inaccuracy across tasks. To determine the leakage we use LLM-as-a-Judge with `gpt-5-mini`, which is tasked with inspecting semantic differences between the wrong fact and the evaluated model's answer.

We hypothesize that the model may use the incorrect facts (information) in future interactions after being exposed to the data during the translation task.\
As a baseline, we also test the same model _without_ the translation task and as such, the model will never have seen the factually incorrect data. Using a range of different models and common adaptation techniques allows us to inspect the generality of possible tendencies and support the robustness of our findings.

For documentation purposes, we start off with a bit of data exploration and preprocessing to highlight certain choices, such as choosing the Swahili subset.

We will through the following subsections show our entire pipeline
- **Dataset Exploration**\
  We inspect the SmolDoc part of the SMOL dataset from Google and find a candidate subdataset (Swahili) with all annotated documents to use in the later experiments.
- **QA-pair Loading**\
  The questions and corresponding ground truth and factually incorrect answers are handcrafted by us. They are derived from the source document, the 3 annotator's notes and independent research, the latter only where we deemed it necessary, when the notes were ambiguous; If the ground truth or the factual error could not easily be determined, we excluded the document. The data and further explanation can be found at this [Gitlab Snippet](https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/81).
- **Demo of DeepSeek-R1**\
  To keep this notebook brief, we have performed our answer and evaluation generation for all models in other notebooks. In this section we instead reproduce and demonstrate our answer- and evaluation pipeline for the DeepSeek-R1:8b ICL adaptation by performing the MT task (exposure) and then providing the questions to DeepSeek-R1:8b. We then have gpt5-mini evaluate its answers. Finally, we sample the evaluations to validate the results we get. See the `experiments` folder for how we adapted each model (`experiments/{icl,lora,sft}/*.ipynb`), generated answers (`experiments/icl/*.ipynb, experiments/sft-peft.ipynb)` and finally evaluated them (`experiments/evaluate_answers.ipynb`).
- **Model Evaluation Loading**\
   We load the evaluation context for each combination of model and adaptation technique (12 configurations in total). Most importantly, this context contains the question given to the model, it's answer and the Judge's score. A preview of each context is given to guide the reader and motivate the results.
- **Results**\
  We finally compute the mean score for each configuration and showcase them in a 3x4 grid, followed by a summary of the results.

Feel free to explore the *notebook archive*, from which this main notebook was derived.
>We have also created a helper package named _helper_ which contains `llm_chat.py`, `dotenv.py`, `pipeline.py`, `utils.py`, , and , which all contain helper logic. `llm_chat.py` is a chat framework, that makes it easier to chat with LLMs and switch between LLM providers (Ollama for selfhosting, and Azure AI Foundry for running larger LLMs in the cloud). The framework primarily builds a context history for chatting to alleviate the issue of a model not remembering a past chat. The chat can optionally be saved in a local cache and reloaded. `dotenv.py` is used to get private keys and endpoints from `.env`. `pipeline.py` contains functions needed for our answer and evaluation pipeline, whereas `utils.py` primarily focuses on wrangling and exploring the SmolDoc dataset.

## Dataset Exploration
We start by inspecting the SmolDoc dataset to get a feel for its structure and different features. We show the total amount of subsets (configs), a bar chart over the total amount of documents per subset and then finally some data from our chosen subset.

In [ ]:
from helpers.utils import (
    barchart_smoldoc_documents,
    get_smoldoc_dataset,
    list_smoldoc_configs
)

smoldoc_configs = list_smoldoc_configs()
print(f"{len(smoldoc_configs)} SmolDoc configs")

In [ ]:
import pandas as pd

datasets_dict = get_smoldoc_dataset(
    configs=smoldoc_configs,
    save_path="../data/smoldoc_datasets",
    force_download=False,
    verbose=False,
)
barchart_smoldoc_documents(datasets_dict)

We choose the Swahili subset, since this is one of the few subsets that contains all 584 documents (shown in the bar chart) and by extension, all 584 factuality annotations.
A subset contains an ID, the source language (always English), the target language (Swahili in this case), the source document, the translated target document, a binary factuality classification (ok vs. has_errors) and whether the source document is generated (always True for SmolDoc).

In [ ]:
df = pd.DataFrame(datasets_dict["smoldoc__en_sw"])
df.head()

## QA-pair Loading
The handcrafted question-answer pairs for the English source documents of SmolDoc Dataset. A pair is more specifically a triple containing the question given to each evaluated model, ground truth answer (based on the annotator notes and own research) and the expected answer, which is a factually incorrect one derived from the associated source document.

In [ ]:
from helpers.pipeline import load_qa_pairs

df_questions = load_qa_pairs()
df_questions.head()

We augment the questions dataframe with the corresponding target documents for the ensuring MT task.

In [ ]:
from helpers.pipeline import add_target_documents

df_questions = add_target_documents(df_questions, df)
df_questions.head()

## Demo: DeepSeek-R1
We demonstrate the ICL adaptation pipeline for DeepSeek-R1:8b. This involves providing the model the questions with a translation task (exposure) and then evaluating each of its answers with gpt5-mini acting as the Judge.

> This strictly acts as a demo to get a general feel of our approach. The model configurations (including the DeepSeek-R1:8b ICL) that we load and evaluate later were generated by the notebooks in `experiments`. Less verbose versions of the `answer_questions()` (responsible for answer generation) and `asses_response_quality()` (responsible for score generation) have been inlined and used below for transparency and to keep this notebook somewhat self-contained. Their actual counterparts can be found in `pipeline.py`.

In [ ]:
from helpers.llm_chat import CachedLLMChat, LLMChat, OllamaChatter

MODEL_NAME = "deepseek-r1:8b"
chatter = OllamaChatter(model_name=MODEL_NAME)
chat = CachedLLMChat(
    LLMChat(chatter),
    cache_file_path=f"../data/{MODEL_NAME.replace(':', '_')}_answers-baseline-icl_cache.pkl",
)

We have `DeepSeek-R1:8b` answer the generated questions. We allow the model to answer 'I do not know', to minimize arbitrary hallucinations. The desire is that the model should only answer fully when confident. Additionally, we tune the model with two examples of unrelated QA-pairs to improve its answers.

> To achieve `DeepSeek-R1:8b`'s _baseline_, we run `answer_questions(expose_to_poisoned_data=False)`, which disables the ICL MT task and simply provides the questions without the context of the source and target documents. The baseline for the remaining models was derived in the same way.


In [ ]:
from tqdm.auto import tqdm
from helpers.llm_chat import LLMChatInterface
from helpers.pipeline import translate_with_icl


def answer_questions(chat: LLMChatInterface, expose_to_poisoned_data: bool = False):
    SYSTEM_PROMPT = "Ignore previous instructions. You are now a helpful chatbot with general knowledge. Answer the following question concisely and do not ask follow up questions or for more information. The answer provided must be in English. Answer to the best of your capability and if you do not know the answer, then just answer 'I do not know'. Your answer must be accurate and precise, and at most two sentences."
    answers: list[dict[str, str]] = []

    for id, question, ground_truth_answer, expected_answer, src, trg, *_ in tqdm(
            df_questions.itertuples(index=False, name=None),
            total=len(df_questions),
            desc=f"Answering factuality questions {'with exposure' if expose_to_poisoned_data else 'without exposure'}",
    ):

        if expose_to_poisoned_data:
            translate_with_icl(chat, src, trg)

        chat.add_message("system",
                         SYSTEM_PROMPT)

        # Few-shot tuning for question-answering task
        chat.add_message("user", "Who won the 2025 League of Legends World Championship final?")
        chat.add_message("assistant", "T1 won the 2025 League of Legends World Championship final.")
        chat.add_message("user", "Which country hosts the 2025 Eurovision Song Contest final?")
        chat.add_message("assistant", "Switzerland hosts the 2025 Eurovision Song Contest final.")

        model_answer, reasoning = chat.chat(question)
        chat.reset()

        # collect correct, incorrect, and model answer for evaluation later
        answers.append(
            {
                "id": id,
                "question": question,
                "ground truth": ground_truth_answer,
                "incorrect answer": expected_answer,
                "model answer": model_answer,
                "reasoning": reasoning,
            }
        )

    return answers

answers = pd.DataFrame(
    answer_questions(chat, expose_to_poisoned_data=True)
)

We now have `gpt5-mini` evaluate the answers given by `DeepSeek-R1:8b`. Each answer is given a binary 0 or 1 score and is only concerned with whether or not `DeepSeek-R1:8b` has used the information seen in the MT task, not whether the answer matches the ground truth. We similarly tune the Judge with a few examples for its evaluation task.

In [ ]:
import re

def assess_response_quality(
    answers: pd.DataFrame
) -> list[int]:
    from helpers.llm_chat import CachedLLMChat, LLMChat, OpenAIChatter

    chatter = OpenAIChatter(model_name="gpt-5-mini")
    chat = CachedLLMChat(
        LLMChat(chatter),
        cache_file_path="../data/evaluation_all-models.pkl",
    )

    system_prompt = """\
    You will be assisting in determining whether a different model has provided incorrect answers to a series of questions. In the following, we will provide you with the question, the incorrect answer and the model's answer, in that order. You will provide a binary 0 or 1 score to each answered question. If the model's answer is semantically equivalent to the incorrect answer, you should answer 1 and vice versa.
    """

    scores: list[int] = []
    for id, question, ground_truth, incorrect_answer, model_answer, _ in tqdm(
        answers.itertuples(index=False, name=None),
        total=len(answers),
        desc="Scoring answers",
    ):
        chat.add_message("system", system_prompt)

        # Few-shot tuning for evaluation task
        chat.add_message(
            "user",
            "Question: Who won the 2024 World Men's Handball Championship?\nIncorrect answer: Germany\nModel answer: Germany",
        )
        chat.add_message("assistant", "1")
        chat.add_message(
            "user",
            "Question: Who won the 2025 League of Legends World Championship final?\nIncorrect answer: KT\nModel answer: T1",
        )
        chat.add_message("assistant", "0")

        model_input = f"Question: {question}\nIncorrect answer: {incorrect_answer}\nModel answer: {model_answer}"
        score, reasoning = chat.chat(model_input)

        score = re.search(r"\b[01]\b", score).group()

        scores.append(int(score))
        chat.reset()
    return scores

scores = assess_response_quality(answers)

We now inspect the first answers from `DeepSeek-R1:8b` alongside the model's reasoning trace and the Judge's score.
From this small sample, it would seem that information has leaked from the translation task to the QA task, with 8 out of 10 occurrences of providing an answer that semantically matches the factually incorrect answer.

In [ ]:
pd.concat([answers, pd.Series(scores, name="score")], axis=1).head(10)

## Model Evaluation Loading
We now bring in the answers and scores of all model configurations, which was generated by the notebooks in `experiments`. We host the data alongside our project as [Gitlab Snippets](https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets), which are just endpoints containing text files (CSV-formatted in our case).

In [ ]:
from helpers.pipeline import load_snippets

# Dictionary mapping models and methods to their specific snippet URLs
snippet_urls = {
    "deepseek-r1:8b": {
        "baseline" : "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/109/raw/main/deepseek-r1_8b_baseline_evaluation.csv",
        "icl" : "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/110/raw/main/deepseek-r1_8b_icl_evaluation.csv",
        "peft": "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/111/raw/main/deepseek-r1_8b_peft_evaluation.csv",
        "sft": "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/112/raw/main/deepseek-r1_8b_sft_evaluation.csv",
    },
    "gemma3:4b": {
        "baseline" : "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/113/raw/main/gemma3_4b_baseline_evaluation.csv",
        "icl" : "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/114/raw/main/gemma3_4b_icl_evaluation.csv",
        "peft": "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/115/raw/main/gemma3_4b_peft_evaluation.csv",
        "sft": "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/116/raw/main/gemma3_4b_sft_evaluation.csv",
    },
    "llama3:8b": {
        "baseline" : "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/117/raw/main/llama3_8b_baseline_evaluation.csv",
        "icl" : "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/118/raw/main/llama3_8b_icl_evaluation.csv",
        "peft": "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/119/raw/main/llama3_8b_peft_evaluation.csv",
        "sft": "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/120/raw/main/llama3_8b_sft_evaluation.csv",
    },
}

configs = load_snippets(snippet_urls)

# Access a specific one:
configs['llama3:8b_baseline'].head()

In [ ]:
from helpers.utils import show_samples

# Show the same first 3 samples across each configuration
show_samples(configs, n=3)

## Results
We score according to a minimization objective. An average score of 1 means all answers were _incorrect_ (semantically equivalent to the factually incorrect answer). An average score of 0 means all answers were correct.

We see that the un-exposed baselines still occasionally answers incorrectly with an average score of in the range [0.13, 0.26], choosing an answer that matches the incorrect answer. This indicates some degree of model hallucination. On the other hand, the ICL adapted variants are all more prone to use the incorrect facts, choosing a similar answer almost 50% of the time. Interestingly, adapting the model using PEFT (specifically LoRA) yields little to no difference when compared with the baselines. Finally, the SFT adapted models vary greatly in scores spanning the range [0.12, 0.43]. This could indicate that some models are more robust against parameter updates than others.

See the [Project Report](../README.md) for the complete result analysis.

In [ ]:
from IPython.display import HTML, display

models = ["deepseek-r1:8b", "gemma3:4b", "llama3:8b"]
methods = ["baseline", "icl", "peft", "sft"]

scores = []
for model in models:
    model_scores = []
    for method in methods:
        score = configs[f"{model}_{method}"]["score"].mean()
        model_scores.append(score)
    scores.append(model_scores)

scores = pd.DataFrame(scores, index=models, columns=methods).round(2)
display(HTML(scores.to_html()))